# Land Price Estimation — Colombo District (`final_dataset.csv`)Replication of the methodology in Naotunna (2022), *A Model for the Estimation of LandPrices in Colombo District using Web Scraped Data* (UCSC), applied to the 2023–2026ikman.lk dataset.**Pipeline (following the dissertation)**1. Load data2. Outlier removal — per GN division, ±2 SD (§3.2.3)3. Log transform of the target (§3.3.3)4. One-hot encoding of `Land_type` (§3.3.2)5. 70/30 train-test split (§3.5.1)6. Scenario 1 — five algorithms, no feature reduction (§4.2.1)7. Scenario 2 — RFECV feature selection (§4.2.2)8. Scenario 3 — RandomizedSearchCV hyperparameter optimisation (§4.2.3)9. Feature importances and diagnostic plots10. Spatial cross-validation check *(addition — not in the original study)*Metrics reported: MSE, RMSE, MAPE, R² (§3.6).

In [ ]:
import osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport warningswarnings.filterwarnings('ignore')from sklearn.model_selection import (train_test_split, RandomizedSearchCV,                                     GroupKFold, cross_val_score)from sklearn.feature_selection import RFECVfrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LinearRegressionfrom sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressorfrom sklearn.svm import SVR, LinearSVRfrom sklearn.metrics import r2_score, mean_squared_errorfrom xgboost import XGBRegressor          # pip install xgboostRANDOM_STATE = 42pd.set_option('display.width', 140)print('Libraries loaded.')

## 1. Load

In [ ]:
CANDIDATES = ['final_dataset.csv', '/mnt/user-data/uploads/final_dataset.csv']DATA_PATH = next(p for p in CANDIDATES if os.path.exists(p))df = pd.read_csv(DATA_PATH, low_memory=False)print(f'Loaded {DATA_PATH}')print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')df.head(3)

## 2. Outlier removal (§3.2.3)The study removes outliers *within each GN division*, not across the dataset as a whole,because price ranges differ enormously between divisions. An observation more than2 standard deviations from its own division's mean is dropped — first on price per perch,then on land size.

In [ ]:
def drop_per_group(d, col, group='Address', k=2):    g = d.groupby(group)[col]    mean, sd = g.transform('mean'), g.transform('std')    keep = sd.isna() | (d[col].sub(mean).abs() <= k * sd)   # sd is NaN for single-listing divisions    return d[keep], (~keep).sum()df, n_price = drop_per_group(df, 'Price per Perch')df, n_size  = drop_per_group(df, 'Land_size(Perches)')print(f'Removed {n_price} price-per-perch outliers')print(f'Removed {n_size} land-size outliers')print(f'Total removed: {n_price + n_size}   Remaining: {len(df):,} rows')

## 3. Target and features (§3.3)Price per perch spans several orders of magnitude, so it is log transformed. `Land_type`is a nominal variable with multi-category values ("Commercial, Residential"), one-hotencoded on the combination strings exactly as the study does.Following Table 3.5, the posted date and price scale are **not** used as predictors.

In [ ]:
# --- target: log transform (§3.3.3)y = np.log(df['Price per Perch'])# --- one-hot encode Land_type on the combination strings (§3.3.2)land_type_dummies = pd.get_dummies(df['Land_type'], prefix='Land_type').astype(int)# --- amenity distance / count variables (Table 3.4)amenity_cols = [c for c in df.columns if c.startswith(('min_dist', 'count_'))]X = pd.concat(    [df[['Land_size(Perches)', 'Distance from fort'] + amenity_cols], land_type_dummies],    axis=1)feature_cols = list(X.columns)print(f'{X.shape[1]} features: 2 property/location + '      f'{len(amenity_cols)} amenity + {land_type_dummies.shape[1]} land-type dummies')print('Any NaNs:', X.isna().sum().sum())

## 4. Train / test split (§3.5.1) and scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.30, random_state=RANDOM_STATE)scaler = StandardScaler().fit(X_train)X_train_s = scaler.transform(X_train)X_test_s  = scaler.transform(X_test)print(f'Train: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}')

## 5. Evaluation helpers (§3.6)MAPE is reported two ways because the dissertation is ambiguous about which scale it used:`MAPE_log` on the log target (what the model actually predicts) and `MAPE_raw` afterexponentiating back to rupees per perch. Quote whichever you use consistently — the twoare not interchangeable.

In [ ]:
def evaluate(model, xtr, xte, name):    model.fit(xtr, y_train)    pred = model.predict(xte)    mse = mean_squared_error(y_test, pred)    return {        'model': name,        'MSE': mse,        'RMSE': np.sqrt(mse),        'MAPE_log': np.mean(np.abs((y_test - pred) / y_test)) * 100,        'MAPE_raw': np.mean(np.abs((np.exp(y_test) - np.exp(pred)) / np.exp(y_test))) * 100,        'R2': r2_score(y_test, pred) * 100,    }def build_models():    return {        'MLR':     LinearRegression(),        'RF':      RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),        'SVR':     SVR(),        'ETR':     ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),        'XGBoost': XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),    }

## 6. Scenario 1 — all features, default hyperparameters (§4.2.1)

In [ ]:
scenario1 = pd.DataFrame(    [evaluate(m, X_train_s, X_test_s, n) for n, m in build_models().items()]).set_index('model')print('--- SCENARIO 1: before feature selection ---')scenario1.round(4)

## 7. Scenario 2 — RFECV feature selection (§4.2.2)The study runs a wrapper method (RFECV) and a filter method (SelectKBest) and keepswhichever performs better; RFECV won for every algorithm, so only RFECV is reproduced here.Two deviations for runtime: `step=2` instead of 1, and `LinearSVR` in place of alinear-kernel `SVR` for the SVR selector — kernel SVR does not expose `coef_` fast enoughto recurse over 37 features. Selection only; the evaluation below still uses a real `SVR`.This cell takes a few minutes.

In [ ]:
selectors = {    'MLR':     LinearRegression(),    'RF':      RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),    'SVR':     LinearSVR(random_state=RANDOM_STATE, max_iter=5000),    'ETR':     ExtraTreesRegressor(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),    'XGBoost': XGBRegressor(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),}kept_features = {}for name, est in selectors.items():    sel = RFECV(est, step=2, cv=3, scoring='r2', n_jobs=-1, min_features_to_select=3)    sel.fit(X_train_s, y_train)    kept_features[name] = [f for f, keep in zip(feature_cols, sel.support_) if keep]    print(f'RFECV {name:8s}: kept {sel.n_features_}/{len(feature_cols)} features')

In [ ]:
rows = []for name, model in build_models().items():    idx = [feature_cols.index(f) for f in kept_features[name]]    rows.append(evaluate(model, X_train_s[:, idx], X_test_s[:, idx], name))scenario2 = pd.DataFrame(rows).set_index('model')print('--- SCENARIO 2: after RFECV ---')scenario2.round(4)

## 8. Scenario 3 — hyperparameter optimisation (§4.2.3)Grids are taken from Tables 4.7, 4.9, 4.11 and 4.13, with `RandomizedSearchCV` as in thestudy. `n_iter=10, cv=3` keeps this to roughly 10–20 minutes; raise `n_iter` if you havetime, and expect the `absolute_error` criterion draws to be slow.Linear regression is skipped: the study tuned `normalize` and `n_jobs`, found noimprovement, and `normalize` no longer exists in current scikit-learn.

In [ ]:
grids = {    'RF': (RandomForestRegressor(random_state=RANDOM_STATE), {        'n_estimators': [75, 100, 150],        'criterion': ['squared_error', 'absolute_error'],        'max_depth': [None, 2, 4, 6],        'max_features': ['sqrt', 'log2', None],        'bootstrap': [True, False],        'min_samples_split': [2, 4, 6],        'min_samples_leaf': [1, 2, 3]}),    'SVR': (SVR(), {        'C': [1.0, 10.0, 20.0, 50.0, 100.0],        'gamma': ['scale', 'auto'],        'kernel': ['linear', 'poly', 'rbf', 'sigmoid']}),    'ETR': (ExtraTreesRegressor(random_state=RANDOM_STATE), {        'n_estimators': [50, 75, 100, 150, 200, 300],        'min_samples_split': [2, 3, 4, 5, 8, 10],        'min_samples_leaf': [1, 2, 3, 4, 6, 8],        'max_features': ['sqrt', 'log2', None],        'criterion': ['squared_error', 'absolute_error']}),    'XGBoost': (XGBRegressor(random_state=RANDOM_STATE, verbosity=0), {        'tree_method': ['auto', 'exact', 'approx', 'hist'],        'n_estimators': [50, 75, 100, 150, 200],        'min_child_weight': [1, 2, 3, 4, 5, 10],        'max_depth': [4, 5, 6, 8, 10, 12, 20],        'booster': ['gbtree', 'gblinear', 'dart']}),}rows, best_models, best_params = [], {}, {}for name, (estimator, grid) in grids.items():    idx = [feature_cols.index(f) for f in kept_features[name]]    search = RandomizedSearchCV(estimator, grid, n_iter=10, cv=3, scoring='r2',                                n_jobs=-1, random_state=RANDOM_STATE)    search.fit(X_train_s[:, idx], y_train)    best_models[name], best_params[name] = search.best_estimator_, search.best_params_    pred = search.best_estimator_.predict(X_test_s[:, idx])    mse = mean_squared_error(y_test, pred)    rows.append({'model': name, 'MSE': mse, 'RMSE': np.sqrt(mse),                 'MAPE_log': np.mean(np.abs((y_test - pred) / y_test)) * 100,                 'MAPE_raw': np.mean(np.abs((np.exp(y_test) - np.exp(pred)) / np.exp(y_test))) * 100,                 'R2': r2_score(y_test, pred) * 100})    print(f'{name:8s} done -> {search.best_params_}')scenario3 = pd.DataFrame(rows).set_index('model')print('\n--- SCENARIO 3: after hyperparameter optimisation ---')scenario3.round(4)

## 9. Comparison across the three scenarios

In [ ]:
comparison = pd.concat(    [scenario1['R2'].rename('Scenario 1 (all features)'),     scenario2['R2'].rename('Scenario 2 (RFECV)'),     scenario3['R2'].rename('Scenario 3 (tuned)')],    axis=1)print('R2 (%) by scenario')comparison.round(2)

In [ ]:
best_name = comparison.max(axis=1).idxmax()print(f'Best algorithm: {best_name}')print(f'Best R2: {comparison.loc[best_name].max():.2f}%')print(f'Features used: {len(kept_features[best_name])}')if best_name in best_params:    print(f'Hyperparameters: {best_params[best_name]}')

## 10. Feature importances (tree models)

In [ ]:
model_for_importance = best_name if best_name in ('RF', 'ETR', 'XGBoost') else 'RF'idx = [feature_cols.index(f) for f in kept_features[model_for_importance]]model = best_models.get(model_for_importance)if model is None or not hasattr(model, 'feature_importances_'):    model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)    model.fit(X_train_s[:, idx], y_train)importances = (pd.Series(model.feature_importances_, index=kept_features[model_for_importance])                 .sort_values(ascending=False))fig, ax = plt.subplots(figsize=(8, 9))importances.sort_values().plot.barh(ax=ax, color='#4C72B0')ax.set_xlabel('Importance')ax.set_title(f'{model_for_importance} feature importances')plt.tight_layout()plt.show()importances.head(15).to_frame('importance')

## 11. Predicted vs. actual

In [ ]:
idx = [feature_cols.index(f) for f in kept_features[best_name]]final_model = best_models.get(best_name, build_models()[best_name])final_model.fit(X_train_s[:, idx], y_train)y_pred = final_model.predict(X_test_s[:, idx])final_r2 = r2_score(y_test, y_pred)fig, axes = plt.subplots(1, 2, figsize=(13, 5))axes[0].scatter(y_test, y_pred, s=6, alpha=0.25, color='#4C72B0')lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]axes[0].plot(lims, lims, 'r--', lw=1)axes[0].set_xlabel('Actual log(Price per Perch)')axes[0].set_ylabel('Predicted')axes[0].set_title(f'{best_name}: Predicted vs Actual (R2 = {final_r2:.3f})')axes[1].hist(y_test - y_pred, bins=60, color='#55A868', edgecolor='white')axes[1].axvline(0, color='r', ls='--', lw=1)axes[1].set_xlabel('Residual (actual - predicted)')axes[1].set_ylabel('Count')axes[1].set_title('Residual distribution')plt.tight_layout()plt.show()

## 12. Spatial cross-validation check *(not in the original study)*Every amenity distance and count is computed from the GN-division centroid, so alllistings in one division share an identical predictor profile. A random split lets themodel memorise "this division's profile means this price". Grouping the folds by divisionmeasures something harder and more useful: prediction in a neighbourhood the model hasnever seen.Report both. The random-split figure is the one comparable to the dissertation; this oneis the defensible estimate of real predictive ability.

In [ ]:
X_all_s = StandardScaler().fit_transform(X)idx = [feature_cols.index(f) for f in kept_features[best_name]]spatial_model = (best_models[best_name] if best_name in best_models                 else build_models()[best_name])spatial_scores = cross_val_score(    spatial_model, X_all_s[:, idx], y,    groups=df['Address_ID'],    cv=GroupKFold(n_splits=5),    scoring='r2', n_jobs=-1,)print('GroupKFold (by GN division) R2 per fold:', np.round(spatial_scores, 3))print(f'Mean spatial R2:  {spatial_scores.mean() * 100:.2f}%')print(f'Random-split R2:  {final_r2 * 100:.2f}%')print(f'Gap: {(final_r2 - spatial_scores.mean()) * 100:.1f} percentage points')

## Notes for the write-up- **Four of the study's variables are missing here.** Both supermarket columns are empty in  the source data and were dropped, and the Class A / Class B government school split could  not be reproduced (the same school pool feeds both, so the pairs were collinear). The  original RF used supermarket distance and count as real predictors, so this is not a  like-for-like feature set.- **The linear baseline is much weaker than the study's** (~68% vs 87.8%). Their 2021 prices  ranged from LKR 4,600 to 85 million per perch — a wide, close-to-linear gradient. This  dataset is compressed and censored at 10 million per perch, which a straight line fits  far less well and which lowers the ceiling for every model.- **The datasets are not comparable in span.** 3,725 listings over two months in 2021 versus  ~6,500 over four years to 2026. More heterogeneity means lower R²; restricting to a  narrower time window would raise it.- **The 2 km radius question.** Table 3.4 states 5 km for schools, universities and  government hospitals, but the study's own published workbook shows no count/distance  overlap beyond 2 km in any of the twelve count columns. This dataset follows the  published data, not the text.- **On the study's reported figures.** The abstract calls 0.098065 the MAE, but Table 4.8  lists it as MSE and its square root is the reported RMSE of 0.313154. It is MSE.  Their row count also does not reconcile: 4,494 published rows minus the stated 278  outliers is 4,216, not the 3,725 modelled.